# Chapter 8 Tutorial — Markov Processes

This notebook is a step-by-step tutorial for Chapter 8, **Markov Processes**.

The chapter extends discrete-time Markov chains to **continuous time**. The main idea is:

> A Markov process is a process whose future after time \(t\), given the present state \(Y_t\), does not depend on the past before \(t\).

The chapter develops:

1. Continuous-time Markov processes and transition functions.
2. Examples: Poisson process, compound Poisson process, Markov chains subordinated to Poisson processes, and \(M/M/1\) queues.
3. Sample-path behavior: continuity, right-continuity, stable/absorbing/instantaneous states.
4. Structure of a Markov process: embedded chain + holding times.
5. Regularity and non-explosion.

We will use `numpy`, `scipy`, and `matplotlib` for computations and simulations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import exp, factorial
from scipy.linalg import expm
from scipy.stats import poisson, expon
rng = np.random.default_rng(12345)

## 1. Markov Processes

Let \(E\) be a countable state space. A stochastic process

\[
Y = \{Y_t : t \in \mathbb{R}_+\}
\]

with values in \(E\) is a **Markov process** if, for all \(t \ge 0\), \(s \ge 0\), and states \(i,j \in E\),

\[
P(Y_{t+s}=j \mid Y_u,\ u \le t) = P(Y_{t+s}=j \mid Y_t).
\]

That is, once the present state \(Y_t\) is known, the earlier path gives no extra information about the future.

### Time-homogeneous Markov process

The process is **time-homogeneous** if

\[
P(Y_{t+s}=j \mid Y_t=i)
\]

depends on \(s\), \(i\), and \(j\), but not on \(t\). Then we define the **transition function**

\[
P_t(i,j) = P(Y_{t+s}=j \mid Y_s=i).
\]

For each fixed \(t\), \(P_t\) is a transition matrix.

### Core laws of transition functions

For all \(t,s \ge 0\),

\[
P_t(i,j) \ge 0,
\]

\[
\sum_j P_t(i,j)=1,
\]

and

\[
\sum_k P_s(i,k)P_t(k,j)=P_{s+t}(i,j).
\]

The last equation is the continuous-time version of the Chapman–Kolmogorov equation.

### Thinking model

A discrete-time Markov chain has one transition matrix \(P\).  
A continuous-time Markov process has a whole family of matrices:

\[
P_0, P_{0.1}, P_{0.2}, \ldots, P_t, \ldots
\]

with the compatibility rule

\[
P_{s+t}=P_sP_t.
\]

So \(P_t\) acts like a matrix-valued exponential clock.

In [ ]:
# A simple finite-state continuous-time Markov chain can be represented by a rate matrix Q.
# Then P_t = exp(tQ), and P_{s+t}=P_s P_t.

Q = np.array([
    [-2.0, 2.0],
    [ 1.0,-1.0]
])

def P(t):
    return expm(t * Q)

s, t = 0.7, 1.3
print("P_s P_t:")
print(P(s) @ P(t))
print("\nP_{s+t}:")
print(P(s+t))
print("\nmax absolute difference:", np.max(np.abs(P(s) @ P(t) - P(s+t))))

### Finite-dimensional distributions

If the initial distribution of \(Y_0\) is \(\pi\), then for

\[
0 \le t_0 < t_1 < \cdots < t_n
\]

and states \(i_0,\ldots,i_n\),

\[
P(Y_{t_0}=i_0,\ldots,Y_{t_n}=i_n)
=
\pi(i_0)
P_{t_1-t_0}(i_0,i_1)
P_{t_2-t_1}(i_1,i_2)
\cdots
P_{t_n-t_{n-1}}(i_{n-1},i_n).
\]

This is exactly like the discrete-time Markov chain formula, except the transition matrix depends on the elapsed time between observations.

In [ ]:
# Example finite-dimensional probability for the 2-state process above.
# P(Y_0=0, Y_0.5=1, Y_1.2=0) with initial distribution pi.
pi = np.array([0.8, 0.2])
prob_path = pi[0] * P(0.5)[0,1] * P(0.7)[1,0]
prob_path

## Example 1.7 — Poisson Process as a Markov Process

Let \(N=\{N_t:t\ge0\}\) be a Poisson process with rate \(\lambda\).

If \(N_t=i\), then after an additional time \(s\), the number of new arrivals is independent of the past and has distribution

\[
N_{t+s}-N_t \sim \mathrm{Poisson}(\lambda s).
\]

Therefore,

\[
P_s(i,j)=
\begin{cases}
e^{-\lambda s}\dfrac{(\lambda s)^{j-i}}{(j-i)!}, & j\ge i,\\[6pt]
0, & j<i.
\end{cases}
\]

The process is Markov because only the current count matters. The previous arrival times do not matter for the future count distribution.

In [ ]:
def poisson_transition(lam, t, i, j):
    if j < i:
        return 0.0
    return poisson.pmf(j-i, lam*t)

lam = 3.0
t = 2.0
i = 4
probs = [poisson_transition(lam, t, i, j) for j in range(0, 20)]

plt.figure(figsize=(8,4))
plt.stem(range(0,20), probs)
plt.xlabel("future count j")
plt.ylabel(r"$P_t(i,j)$")
plt.title(r"Poisson process transition probabilities from $i=4$, $\lambda=3$, $t=2$")
plt.show()

### Simulating sample paths of a Poisson process

A Poisson process can be built by exponential waiting times:

\[
W_1,W_2,\ldots \stackrel{iid}{\sim} \mathrm{Exponential}(\lambda),
\]

\[
T_n = W_1+\cdots+W_n.
\]

Then

\[
N_t = \max\{n:T_n\le t\}.
\]

The path is a right-continuous step function with unit jumps.

In [ ]:
def simulate_poisson_path(lam=2.0, T=5.0, rng=rng):
    waits = []
    t = 0.0
    while t < T:
        w = rng.exponential(1/lam)
        waits.append(w)
        t += w
    arrivals = np.cumsum(waits)
    arrivals = arrivals[arrivals <= T]
    times = np.r_[0, arrivals, T]
    counts = np.r_[0, np.arange(1, len(arrivals)+1), len(arrivals)]
    return times, counts, arrivals

times, counts, arrivals = simulate_poisson_path(lam=2.0, T=5.0)

plt.figure(figsize=(8,4))
plt.step(times, counts, where="post")
plt.scatter(arrivals, np.arange(1, len(arrivals)+1))
plt.xlabel("t")
plt.ylabel(r"$N_t$")
plt.title("A simulated Poisson process path")
plt.show()

## Example 1.8 — Compound Poisson Process

Let \(Y_t\) be a **compound Poisson process**:

\[
Y_t = X_1+\cdots+X_{N_t},
\]

where \(N_t\) is Poisson and the jumps \(X_1,X_2,\ldots\) are iid integer-valued random variables.

If \(Y_t=i\), then after time \(s\),

\[
Y_{t+s}-Y_t = X_{N_t+1}+\cdots+X_{N_{t+s}},
\]

which depends only on the new jumps after time \(t\). Therefore \(Y\) is Markov.

Its transition probabilities have the convolution form

\[
P_s(i,j)=p_s(j-i),
\]

where \(p_s(k)=P(Y_s-Y_0=k)\).

In [ ]:
# Compound Poisson simulation: jumps are +1 with prob .7 and -1 with prob .3.
def simulate_compound_poisson(lam=2.0, T=5.0, p_up=0.7, rng=rng):
    times, counts, arrivals = simulate_poisson_path(lam, T, rng)
    jumps = rng.choice([1, -1], size=len(arrivals), p=[p_up, 1-p_up])
    values = np.r_[0, np.cumsum(jumps)]
    plot_times = np.r_[0, arrivals, T]
    plot_values = np.r_[0, values[1:], values[-1] if len(values) else 0]
    return plot_times, plot_values, arrivals, jumps

times, values, arrivals, jumps = simulate_compound_poisson()

plt.figure(figsize=(8,4))
plt.step(times, values, where="post")
plt.scatter(arrivals, values[1:])
plt.xlabel("t")
plt.ylabel(r"$Y_t$")
plt.title("Compound Poisson process with +1/-1 jumps")
plt.show()

## Example 1.9 — Markov Chain Subordinated to a Poisson Process

Let \(X_0,X_1,\ldots\) be a discrete-time Markov chain with transition matrix \(K\).  
Let \(N_t\) be an independent Poisson process with rate \(\lambda\). Define

\[
Y_t = X_{N_t}.
\]

This means: the chain \(X\) only moves when the Poisson clock rings.

Given \(Y_s=i\), the number of transitions between \(s\) and \(s+t\) is Poisson with mean \(\lambda t\). If exactly \(n\) transitions happen, the transition matrix is \(K^n\). Therefore

\[
P_t = \sum_{n=0}^{\infty} e^{-\lambda t}\frac{(\lambda t)^n}{n!}K^n.
\]

This is the matrix exponential formula

\[
P_t = e^{\lambda t(K-I)}.
\]

In [ ]:
K = np.array([
    [0.5, 0.5],
    [0.3, 0.7]
])
lam = 1.5

def subordinated_transition_series(t, K, lam, n_terms=50):
    Psum = np.zeros_like(K, dtype=float)
    Kn = np.eye(K.shape[0])
    for n in range(n_terms):
        if n > 0:
            Kn = Kn @ K
        Psum += poisson.pmf(n, lam*t) * Kn
    return Psum

def subordinated_transition_expm(t, K, lam):
    return expm(lam * t * (K - np.eye(K.shape[0])))

t = 2.0
print("Series:")
print(subordinated_transition_series(t, K, lam))
print("\nMatrix exponential:")
print(subordinated_transition_expm(t, K, lam))

### Concrete version of Example 1.9

The chapter uses the two-state Markov chain from an earlier example with

\[
K =
\begin{pmatrix}
\frac12 & \frac12 \\
\frac35 & \frac25
\end{pmatrix}
\]

and \(\lambda=15\). Then

\[
P_t = \sum_{n=0}^\infty e^{-15t}\frac{(15t)^n}{n!}K^n.
\]

Computing via the matrix exponential is usually the simplest method.

In [ ]:
K = np.array([
    [1/2, 1/2],
    [3/5, 2/5]
], dtype=float)
lam = 15.0

for t in [0.01, 0.1, 1.0]:
    print(f"t = {t}")
    print(expm(lam * t * (K - np.eye(2))))
    print()

## Example 1.10 — \(M/M/1\) Queueing System

An \(M/M/1\) queue has:

- Poisson arrivals with rate \(\lambda\),
- exponential service times with rate \(\mu\),
- one server,
- state \(Y_t =\) number of customers in the system at time \(t\).

The process is Markov because:

1. Poisson arrivals have independent increments.
2. Exponential service times are memoryless.
3. Therefore, given the current queue length, the future does not need earlier history.

The transition function is not usually written in a simple closed form, but the process is computationally manageable through its generator:

\[
Q(i,i+1)=\lambda,
\]

\[
Q(i,i-1)=\mu \quad (i\ge1),
\]

\[
Q(i,i)=-(\lambda+\mu) \quad (i\ge1),
\]

\[
Q(0,0)=-\lambda.
\]

In [ ]:
def mm1_generator(lam, mu, max_state):
    Q = np.zeros((max_state+1, max_state+1))
    for i in range(max_state+1):
        if i < max_state:
            Q[i, i+1] = lam
        if i > 0:
            Q[i, i-1] = mu
        Q[i, i] = -Q[i].sum()
    return Q

lam, mu = 2.0, 3.0
Q = mm1_generator(lam, mu, max_state=10)
Pt = expm(1.0 * Q)
Pt[0, :8]

In [ ]:
# Simulate an M/M/1 queue path.
def simulate_mm1(lam=2.0, mu=3.0, T=20.0, rng=rng):
    t = 0.0
    y = 0
    times = [0.0]
    states = [y]
    while t < T:
        if y == 0:
            rate = lam
            dt = rng.exponential(1/rate)
            event = "arrival"
        else:
            rate = lam + mu
            dt = rng.exponential(1/rate)
            event = "arrival" if rng.random() < lam/rate else "departure"
        t += dt
        if t > T:
            break
        if event == "arrival":
            y += 1
        else:
            y -= 1
        times.append(t)
        states.append(y)
    times.append(T)
    states.append(states[-1])
    return np.array(times), np.array(states)

times, states = simulate_mm1(lam=2.0, mu=3.0, T=20.0)

plt.figure(figsize=(9,4))
plt.step(times, states, where="post")
plt.xlabel("t")
plt.ylabel("queue length")
plt.title(r"Simulated $M/M/1$ queue, $\lambda=2$, $\mu=3$")
plt.show()

## Strong Markov Property

The chapter states continuous-time analogues of the strong Markov property.

If \(T\) is a stopping time and \(Y_T=i\), then after time \(T\), the process behaves like a fresh Markov process started at \(i\).

For \(s\ge0\),

\[
P(Y_{T+s}=j \mid \mathcal{F}_T) = P_s(i,j)
\quad\text{on } \{Y_T=i\}.
\]

A useful computational form is:

\[
E[f(Y_{T+s_1},\ldots,Y_{T+s_n}) \mid \mathcal{F}_T]
=
g(Y_T),
\]

where

\[
g(i)=E_i[f(Y_{s_1},\ldots,Y_{s_n})].
\]

### Thinking model

A stopping time is a time whose occurrence can be decided from the past and present, not from the future.  
At such a time, the process probabilistically restarts from its current state.

## 2. Sample Path Behavior

Continuous-time processes can have complicated paths. The chapter introduces restrictions that make paths manageable.

### Left limits

For a sample path \(t\mapsto Y_t(\omega)\), a left limit at \(t\) means

\[
\lim_{s \uparrow t} Y_s(\omega)
\]

exists.

### Standard transition function

A transition function \((P_t)\) is called **standard** if

\[
\lim_{t\downarrow0}P_t(i,i)=1
\]

for every \(i\).

Intuition: over a very small time interval, the process is very likely to remain where it is.

### Stochastic continuity

A process \(Y\) is **stochastically continuous** if, for every \(t>0\),

\[
\lim_{s\to t}P(Y_s\ne Y_t)=0.
\]

For Markov processes with standard transition functions, this is equivalent to:

\[
\lim_{s\to t}P_s(i,j)=P_t(i,j).
\]

But stochastic continuity does **not** imply path continuity.  
Poisson process paths are stochastically continuous but have jumps.

In [ ]:
# Poisson process: probability of a jump in a small interval h is 1 - exp(-lambda*h)
lam = 2.0
hs = np.linspace(0, 0.2, 100)
jump_probs = 1 - np.exp(-lam * hs)

plt.figure(figsize=(7,4))
plt.plot(hs, jump_probs)
plt.xlabel("h")
plt.ylabel(r"$P(N_{t+h}\ne N_t)$")
plt.title("Stochastic continuity of Poisson process")
plt.show()

## Modifications and the cemetery state \(\Delta\)

The chapter explains that paths may not have nice pointwise limits everywhere. One can modify the process on a null set and add a new state \(\Delta\) to handle pathological cases.

The new state \(\Delta\) is ordered after all original states. If a path loses a well-defined limit, it can be sent to \(\Delta\).

This is a technical device to obtain nicer sample paths without changing probabilities of observable events.

## Holding Times

For a state \(i\), define the holding time

\[
W_i(\omega)=\inf\{t\ge0:Y_t(\omega)\ne i\}
\]

when \(Y_0=i\).

The key theorem says:

\[
P(W_i>w\mid Y_0=i)=e^{-\lambda(i)w}
\]

for some \(\lambda(i)\in[0,\infty]\).

So the holding time in a state is exponential, unless the state is absorbing or instantaneous.

### Absorbing, stable, and instantaneous states

A state \(i\) is:

- **absorbing** if \(\lambda(i)=0\). Once entered, it is never left.
- **stable** if \(0<\lambda(i)<\infty\). The process remains there for a positive exponential time.
- **instantaneous** if \(\lambda(i)=\infty\). The process leaves immediately.

For a stable state,

\[
P(W_i>w)=e^{-\lambda(i)w}.
\]

For an absorbing state,

\[
P(W_i=\infty)=1.
\]

For an instantaneous state,

\[
P(W_i=0)=1.
\]

In [ ]:
# Holding time distributions for stable states
rates = [0.5, 1.0, 3.0]
w = np.linspace(0, 5, 200)

plt.figure(figsize=(7,4))
for r in rates:
    plt.plot(w, np.exp(-r*w), label=fr"$\lambda={r}$")
plt.xlabel("w")
plt.ylabel(r"$P(W_i>w)$")
plt.title("Survival functions of exponential holding times")
plt.legend()
plt.show()

### Finite state spaces have no instantaneous states

The chapter proves that if \(E\) is finite, then no state is instantaneous.

Intuition: if the process is in a finite state space and leaves states infinitely fast, it would have to make too many jumps in too little time. Standardness prevents this.

## Times spent in a stable state

For a stable state \(i\), define

\[
G_i(\omega)=\{t:Y_t(\omega)=i\}.
\]

The theorem says that almost surely \(G_i\) is a union of intervals of positive length, each of the form

\[
[s,t)
\]

possibly with \(t=\infty\).

So stable-state paths are made of flat plateaus, not isolated time points.

## 3. Structure of a Markov Process

Assume all states are stable.

Then a Markov process can be decomposed into:

1. an embedded discrete-time Markov chain of visited states,
2. exponential holding times in those states.

Define transition times:

\[
T_0=0,
\]

\[
T_{n+1}=T_n+W_n,
\]

and embedded states:

\[
X_n=Y_{T_n}.
\]

Then \(X_0,X_1,\ldots\) is a Markov chain.

### Key structural theorem

There exist:

- a discrete transition matrix \(Q(i,j)\),
- holding rates \(\lambda(i)\),

such that

\[
P(X_{n+1}=j,\ T_{n+1}-T_n>w \mid X_0,\ldots,X_n,T_0,\ldots,T_n)
=
Q(i,j)e^{-\lambda(i)w}
\]

when \(X_n=i\).

This says:

1. The next state depends only on the current state \(i\).
2. The holding time in \(i\) is exponential with rate \(\lambda(i)\).
3. Conditional on \(i\), the next state and holding time separate in this formula.

For \(j=i\), usually \(Q(i,i)=0\) for a stable non-absorbing state; the process jumps to a different state.

### Constructing a continuous-time Markov chain from \(Q\) and \(\lambda\)

Given:

\[
\lambda(i)>0
\]

and a transition matrix \(Q\) with \(Q(i,i)=0\), construct the process as follows:

1. Start in state \(X_0\).
2. Stay there for \(W_0\sim \mathrm{Exponential}(\lambda(X_0))\).
3. Jump to \(X_1\sim Q(X_0,\cdot)\).
4. Repeat.

This is the operational model of a continuous-time Markov chain.

In [ ]:
def simulate_ctmc(Qjump, rates, start=0, T=20.0, rng=rng):
    state = start
    t = 0.0
    times = [0.0]
    states = [state]
    n = len(rates)
    while t < T:
        rate = rates[state]
        if rate == 0:
            break
        dt = rng.exponential(1/rate)
        t += dt
        if t > T:
            break
        state = rng.choice(n, p=Qjump[state])
        times.append(t)
        states.append(state)
    times.append(T)
    states.append(states[-1])
    return np.array(times), np.array(states)

Qjump = np.array([
    [0.0, 0.7, 0.3],
    [0.4, 0.0, 0.6],
    [0.2, 0.8, 0.0]
])
rates = np.array([1.0, 2.5, 0.8])

times, states = simulate_ctmc(Qjump, rates, start=0, T=15)

plt.figure(figsize=(9,4))
plt.step(times, states, where="post")
plt.yticks([0,1,2])
plt.xlabel("t")
plt.ylabel("state")
plt.title("Simulated continuous-time Markov chain from embedded chain + holding rates")
plt.show()

## Relation to the generator matrix

For stable states, the generator matrix \(A\) is

\[
A(i,j)=\lambda(i)Q(i,j), \quad j\ne i,
\]

and

\[
A(i,i)=-\lambda(i).
\]

Then

\[
P_t = e^{tA}.
\]

This is the finite-state computational workhorse.

In [ ]:
def generator_from_jump_rates(Qjump, rates):
    A = rates[:, None] * Qjump
    np.fill_diagonal(A, -rates)
    return A

A = generator_from_jump_rates(Qjump, rates)
print(A)
print("row sums:", A.sum(axis=1))

Pt = expm(2.0 * A)
print("\nP_2:")
print(Pt)
print("row sums:", Pt.sum(axis=1))

## Example 3.12 — Poisson process structure

For a Poisson process \(N_t\):

- the state is the current count \(i\),
- the next state is always \(i+1\),
- the holding rate is constant \(\lambda\).

Thus

\[
\lambda(i)=\lambda,
\]

\[
Q(i,i+1)=1,
\]

and all other \(Q(i,j)=0\).

So the Poisson process is the simplest pure-birth Markov process.

In [ ]:
# Simulate a pure-birth process with constant rate lambda: this is a Poisson process.
def simulate_pure_birth(lam=2.0, T=5.0, rng=rng):
    t = 0.0
    state = 0
    times = [0.0]
    states = [0]
    while t < T:
        t += rng.exponential(1/lam)
        if t > T:
            break
        state += 1
        times.append(t)
        states.append(state)
    times.append(T)
    states.append(state)
    return np.array(times), np.array(states)

times, states = simulate_pure_birth(lam=2.0, T=5.0)
plt.figure(figsize=(8,4))
plt.step(times, states, where="post")
plt.xlabel("t")
plt.ylabel("state")
plt.title("Pure-birth representation of a Poisson process")
plt.show()

## \(M/M/1\) Queue Structure

For the \(M/M/1\) queue:

- From state \(0\), only arrivals can occur.
- From state \(i\ge1\), either an arrival or departure occurs.

The holding rates are

\[
\lambda(0)=a,
\]

\[
\lambda(i)=a+b,\quad i\ge1,
\]

where \(a\) is arrival rate and \(b\) is service rate.

The embedded jump probabilities are

\[
Q(i,i+1)=\frac{a}{a+b},\quad i\ge1,
\]

\[
Q(i,i-1)=\frac{b}{a+b},\quad i\ge1,
\]

and

\[
Q(0,1)=1.
\]

This separates the queue into two pieces:

1. how long until the next event,
2. whether the event is arrival or departure.

In [ ]:
def mm1_jump_matrix(a, b, max_state):
    Qj = np.zeros((max_state+1, max_state+1))
    Qj[0,1] = 1.0
    for i in range(1, max_state):
        Qj[i,i+1] = a/(a+b)
        Qj[i,i-1] = b/(a+b)
    # boundary at max_state: approximate by not allowing upward jump
    Qj[max_state,max_state-1] = 1.0
    return Qj

a, b = 2.0, 3.0
max_state = 8
Qj = mm1_jump_matrix(a,b,max_state)
rates = np.array([a] + [a+b]*max_state)
A_mm1 = generator_from_jump_rates(Qj, rates)
A_mm1

## Absorbing states in the structural representation

If a state \(i\) is absorbing, then

\[
\lambda(i)=0.
\]

The process stays in \(i\) forever once it reaches \(i\).

In the generator matrix, this means the entire \(i\)-th row is zero.

In [ ]:
# CTMC with state 2 absorbing
Qjump_abs = np.array([
    [0.0, 0.6, 0.4],
    [0.7, 0.0, 0.3],
    [0.0, 0.0, 1.0]  # arbitrary; rate is zero, so row will be zero in generator
])
rates_abs = np.array([1.0, 1.5, 0.0])
A_abs = generator_from_jump_rates(Qjump_abs, rates_abs)
A_abs

## Regular Markov Processes

The construction by embedded states and holding times can fail if the process makes infinitely many jumps in finite time. This is called **explosion**.

Define

\[
\zeta(\omega)=\sup_n T_n(\omega).
\]

The process is called **regular** if

1. \(t\mapsto Y_t(\omega)\) is right-continuous almost surely,
2. \(\zeta(\omega)=+\infty\) almost surely.

The second condition means: no explosion in finite time.

### Sufficient condition for regularity

If

\[
\lambda(i)\le c
\]

for all states \(i\), for some finite constant \(c\), then the process is regular.

Intuition: if no state has an arbitrarily high jump rate, then the process cannot squeeze infinitely many jumps into a finite interval.

In [ ]:
# Explosion intuition:
# If rates grow extremely fast, waiting times can become summable.
# For a pure birth process with lambda_n = n^2, the expected total time to infinity is sum 1/n^2 < infinity.
# For lambda_n = n, sum 1/n diverges.

N = 10000
sum_linear = np.sum(1/np.arange(1,N+1))
sum_quadratic = np.sum(1/(np.arange(1,N+1)**2))

sum_linear, sum_quadratic, np.pi**2/6

The previous computation shows the core difference:

\[
\sum_{n=1}^{\infty}\frac1n=\infty,
\]

but

\[
\sum_{n=1}^{\infty}\frac1{n^2}<\infty.
\]

For a pure-birth process with rates \(\lambda(n)=n^2\), the expected total time to make infinitely many jumps is finite. This indicates possible explosion.

For bounded rates, this cannot happen.

In [ ]:
def simulate_birth_explosion(rates_fn, max_jumps=10000, rng=rng):
    waits = np.array([rng.exponential(1/rates_fn(n)) for n in range(1, max_jumps+1)])
    return waits.cumsum()

cum_linear = simulate_birth_explosion(lambda n: n, max_jumps=5000)
cum_quad = simulate_birth_explosion(lambda n: n*n, max_jumps=5000)

plt.figure(figsize=(8,4))
plt.plot(cum_linear[:1000], label=r"$\lambda_n=n$")
plt.plot(cum_quad[:1000], label=r"$\lambda_n=n^2$")
plt.xlabel("number of jumps")
plt.ylabel("cumulative time")
plt.title("Explosion intuition for pure-birth processes")
plt.legend()
plt.show()

print("time after 5000 jumps, lambda_n=n:", cum_linear[-1])
print("time after 5000 jumps, lambda_n=n^2:", cum_quad[-1])

## Summary of the Chapter

### Main conceptual progression

1. **Markov process**: continuous-time analogue of a Markov chain.
2. **Transition function**: \(P_t(i,j)\), a family of transition matrices.
3. **Chapman–Kolmogorov**:

   \[
   P_{s+t}=P_sP_t.
   \]

4. **Examples**:
   - Poisson process.
   - Compound Poisson process.
   - Discrete Markov chain run on a Poisson clock.
   - \(M/M/1\) queue.

5. **Sample paths**:
   - Stochastic continuity does not mean continuous paths.
   - Paths may have jumps.
   - Stable states have positive holding intervals.
   - Absorbing states never leave.
   - Instantaneous states are left immediately.

6. **Structural representation**:
   - embedded Markov chain \(X_n\),
   - exponential holding times,
   - rates \(\lambda(i)\),
   - jump matrix \(Q(i,j)\).

7. **Generator representation**:

   \[
   A(i,j)=\lambda(i)Q(i,j),\quad i\ne j,
   \]

   \[
   A(i,i)=-\lambda(i),
   \]

   and in finite state spaces

   \[
   P_t=e^{tA}.
   \]

8. **Regularity**:
   - no explosion,
   - right-continuous paths,
   - bounded rates imply regularity.

### Big thinking model

A continuous-time Markov chain is:

> a discrete Markov chain plus exponential clocks attached to states.

The embedded chain tells **where to jump next**.  
The exponential clock tells **when to jump**.

## Exercises for self-check

1. For a Poisson process with rate \(\lambda=4\), compute

   \[
   P(N_3=7\mid N_1=2).
   \]

2. For a subordinated Markov chain with

   \[
   K=\begin{pmatrix}0.8&0.2\\0.1&0.9\end{pmatrix},
   \quad \lambda=5,
   \]

   compute \(P_1\).

3. Simulate an \(M/M/1\) queue with \(\lambda=2\), \(\mu=5\). Estimate the long-run average queue length.

4. Given embedded jump matrix

   \[
   Q=\begin{pmatrix}
   0&1\\
   1&0
   \end{pmatrix}
   \]

   and rates \(\lambda(0)=2,\lambda(1)=3\), compute the generator \(A\) and \(P_1=e^A\).

5. Explain why bounded holding rates prevent explosion.

In [ ]:
# Exercise 1 solution
lam = 4
answer1 = poisson.pmf(5, lam*2)  # N3-N1 = 5 over length 2
answer1

In [ ]:
# Exercise 2 solution
K = np.array([[0.8,0.2],[0.1,0.9]])
lam = 5
P1 = expm(lam * (K - np.eye(2)))
P1

In [ ]:
# Exercise 4 solution
Qjump = np.array([[0,1],[1,0]], dtype=float)
rates = np.array([2,3], dtype=float)
A = generator_from_jump_rates(Qjump, rates)
P1 = expm(A)
A, P1